# Simulador de personas da Fornada

Cinco personas, cada uma com um LLM no papel de cliente, conversam com a atendente pela API (`POST /conversations/{id}/messages`). Este notebook só mostra e dispara; a lógica fica no pacote `fornada_simulator`.

**Antes de começar:** a API precisa estar rodando (`uv run task dev` em `apps/api`) e a chave do provedor em `SIMULATOR__*` no `.env` da raiz. O kernel é o `.venv` da raiz (`uv sync --group notebook`).

In [ ]:
from pathlib import Path

import httpx
import pandas as pd

from fornada_simulator.model import build_chat_model
from fornada_simulator.personas import PERSONAS
from fornada_simulator.settings import get_settings
from fornada_simulator.simulate import new_run_id, run_batch, run_conversation
from fornada_simulator.summary import format_transcript, summarize
from fornada_simulator.transcript import read_jsonl

settings = get_settings().simulator
RUNS_DIR = Path("../runs")

health = httpx.get(f"{settings.api_url}/health/db", timeout=5)
print(f"API {settings.api_url}: HTTP {health.status_code}")
print(f"Modelo do simulador: {settings.provider} / {settings.active.model}")

## As personas

Cada persona tem vários objetivos; num lote, cada uma roda com um objetivo diferente.

In [ ]:
pd.set_option("display.max_colwidth", 120)
pd.DataFrame([{"persona": p.id, "objetivo": goal} for p in PERSONAS for goal in p.goals])

## Uma conversa, turno a turno

Troque `persona` e `goal_index` para ver outra. O fim vem de `[FIM]` na mensagem da persona ou do limite de turnos.

In [ ]:
persona = PERSONAS[0]
goal_index = 0

async with httpx.AsyncClient(base_url=settings.api_url, timeout=120) as http:
    transcript = await run_conversation(
        persona,
        persona.goals[goal_index],
        http,
        build_chat_model(settings),
        max_turns=settings.max_turns,
        run_id="notebook",
    )
print(format_transcript(transcript))

## Um lote

`runs_per_persona` conversas por persona (padrão 4, 20 no total). Cada conversa é gravada em `apps/simulator/runs/<run_id>.jsonl` assim que termina.

In [ ]:
run_id = new_run_id()
transcripts = await run_batch(
    settings, build_chat_model(settings), runs_dir=RUNS_DIR, run_id=run_id
)
print(f"{len(transcripts)} conversas em {RUNS_DIR / (run_id + '.jsonl')}")

## Resumo por persona

In [ ]:
def summary_table(transcripts):
    return pd.DataFrame(
        [
            {
                "persona": row.persona_id,
                "conversas": row.conversations,
                **row.outcomes,
                "turnos (média)": row.avg_turns,
                "segundos (média)": row.avg_seconds,
            }
            for row in summarize(transcripts)
        ]
    )


summary_table(transcripts)

## Ler um lote anterior

In [ ]:
latest = sorted(RUNS_DIR.glob("*.jsonl"))[-1]
earlier = read_jsonl(latest)
print(latest.name)
summary_table(earlier)

In [ ]:
# Leia uma conversa do lote (troque o índice).
print(format_transcript(earlier[0]))